# Ring-Gemma4 Depression PoC — End-to-End Demo

This notebook demonstrates the full multimodal pipeline:

1. Generate synthetic PPG signals
2. Preprocess and segment
3. Encode with ResNet-1D
4. Project into LLM token space
5. Combine with EHR text tokens
6. Predict depression risk
7. Visualize projector output with PCA

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

from src.preprocessing import generate_synthetic_ppg, preprocess_ppg
from src.encoder import get_encoder
from src.projector import SensorProjector
from src.model import RingGemmaModel
from src.dataset import DepressionDataset

print('All imports successful!')

## 1. Generate and Preprocess Synthetic PPG

In [ ]:
# Generate a single synthetic PPG signal
signals = generate_synthetic_ppg(n_samples=1, duration=60.0, fs=125.0)
raw_signal, fs = signals[0]

# Plot raw signal (first 5 seconds)
t = np.arange(len(raw_signal)) / fs
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

axes[0].plot(t[:625], raw_signal[:625])
axes[0].set_title('Raw Synthetic PPG (5 seconds)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

# Preprocess
segments = preprocess_ppg(raw_signal, fs_input=fs)
print(f'Preprocessed segments shape: {segments.shape}')
print(f'  → {segments.shape[0]} segments of {segments.shape[2]} samples each')

# Plot first preprocessed segment
axes[1].plot(np.arange(segments.shape[2]) / 125.0, segments[0, 0].numpy())
axes[1].set_title('Preprocessed Segment (z-scored, filtered)')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Normalized Amplitude')

plt.tight_layout()
plt.show()

## 2. Encode with ResNet-1D

In [ ]:
encoder = get_encoder(use_papagei=False)
print(f'Encoder: ResNet-1D, output_dim = {encoder.output_dim}')

with torch.no_grad():
    embeddings = encoder(segments)  # (N_segments, 512)

print(f'Embeddings shape: {embeddings.shape}')
print(f'Embedding stats: mean={embeddings.mean():.4f}, std={embeddings.std():.4f}')

## 3. Project into LLM Token Space

In [ ]:
projector = SensorProjector(encoder_dim=512, llm_dim=2048, n_tokens=16)
n_params = sum(p.numel() for p in projector.parameters() if p.requires_grad)
print(f'SensorProjector: {n_params:,} trainable parameters')

# Add batch dimension
sensor_tokens = projector(embeddings.unsqueeze(0))  # (1, 16, 2048)
print(f'Sensor tokens shape: {sensor_tokens.shape}')
print(f'  → {sensor_tokens.shape[1]} virtual tokens in {sensor_tokens.shape[2]}-dim LLM space')

## 4. Full Model Forward Pass

In [ ]:
# Create model with MockLLM (no GPU needed)
model = RingGemmaModel(use_real_llm=False, device='cpu')
model.eval()

# Sample input
ppg_input = segments.unsqueeze(0)  # (1, N_segments, 1, 1250)
ehr_text = ["Patient reports persistent low mood, fatigue, and sleep disturbances."]

with torch.no_grad():
    result = model(ppg_input, ehr_text)

logits = result['logits']
probs = torch.softmax(logits, dim=-1)
pred = logits.argmax(dim=-1).item()

print(f'Logits: {logits.squeeze().numpy()}')
print(f'Probabilities: No depression={probs[0, 0]:.3f}, Depression={probs[0, 1]:.3f}')
print(f'Prediction: {"Depression" if pred == 1 else "No depression"}')
print()
print('Note: This is an untrained model with synthetic data — predictions are random.')

## 5. PCA Visualization of Sensor Tokens

Visualize how the projector maps PPG segment embeddings into the LLM token space.
We generate multiple samples and plot their sensor tokens after PCA reduction.

In [ ]:
# Generate a small dataset
dataset = DepressionDataset.create_synthetic(n_samples=20)

all_tokens = []
all_labels = []

model.eval()
with torch.no_grad():
    for i in range(len(dataset)):
        sample = dataset[i]
        ppg = sample['ppg_segments'].unsqueeze(0)  # (1, N, 1, 1250)
        text = [sample['ehr_text']]
        label = sample['label']
        
        # Get sensor tokens from projector
        n_seg = ppg.size(1)
        flat = ppg.view(n_seg, 1, -1)
        enc = model.encoder(flat).unsqueeze(0)
        tokens = model.projector(enc)  # (1, 16, 2048)
        
        # Average across tokens for a single point per sample
        mean_token = tokens.mean(dim=1).squeeze().numpy()  # (2048,)
        all_tokens.append(mean_token)
        all_labels.append(label)

all_tokens = np.array(all_tokens)
all_labels = np.array(all_labels)

# PCA to 2D
pca = PCA(n_components=2)
tokens_2d = pca.fit_transform(all_tokens)

fig, ax = plt.subplots(figsize=(8, 6))
for label, name, color in [(0, 'No Depression', 'steelblue'), (1, 'Depression', 'tomato')]:
    mask = all_labels == label
    ax.scatter(tokens_2d[mask, 0], tokens_2d[mask, 1], c=color, label=name,
               alpha=0.7, s=80, edgecolors='white', linewidth=0.5)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA of Projected Sensor Tokens (Untrained Model)')
ax.legend()
plt.tight_layout()
plt.show()

print('Note: With an untrained model, classes overlap.')
print('After training, sensor tokens should separate by class.')

## Summary

This demo showed the complete pipeline:

```
Raw PPG → Preprocess → ResNet-1D Encoder → Projector → [sensor_tokens || text_tokens] → LLM → Classifier
```

Key observations:
- The ResNet-1D encoder produces 512-dim embeddings per PPG segment
- The SensorProjector pools and projects these into 16 virtual tokens in the LLM's 2048-dim space
- These are concatenated with tokenized EHR text and fed through the LLM
- The first hidden state is used for binary depression classification

To train on real data, replace the synthetic generator with actual PPG + EHR datasets.